# Image Compression and Pattern Detection Using Eigenvalues and Orthogonal Projections

**Linear Algebra Mini Project**

---

## Overview

This notebook implements two core applications of linear algebra:

1. **SVD-Based Image Compression** — Represent an image using only the top-k singular values/vectors, dramatically reducing storage while preserving visual quality.
2. **Eigenfaces for Pattern Detection** — Use PCA (via eigendecomposition of the covariance matrix) to find the dominant "face patterns" across a dataset, then reconstruct faces as orthogonal projections onto the eigenface subspace.

**Key Linear Algebra concepts used:**
- Singular Value Decomposition (SVD): $A = U\Sigma V^T$
- Eigendecomposition of covariance matrix: $C = Q\Lambda Q^T$
- Rank-k approximation (best low-rank approximation by Eckart–Young theorem)
- Orthogonal projection: $\hat{x} = QQ^T x$
- Frobenius norm for reconstruction error

---

## Section 1: Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

# Try to load Olivetti faces; fall back to synthetic data if offline
try:
    from sklearn.datasets import fetch_olivetti_faces
    data = fetch_olivetti_faces(shuffle=True, random_state=42)
    faces = data.images          # shape: (400, 64, 64), values in [0,1]
    print("Loaded Olivetti Faces dataset")
except Exception:
    # Synthetic low-rank face-like data (works offline)
    np.random.seed(42)
    n, h, w = 400, 64, 64
    U = np.random.randn(n, 20)
    V = np.random.randn(20, h * w)
    X = (U @ V).reshape(n, h, w)
    X -= X.min(axis=(1,2), keepdims=True)
    X /= X.max(axis=(1,2), keepdims=True) + 1e-8
    faces = X
    print("Using synthetic face-like dataset (offline mode)")

n_samples, H, W = faces.shape
print(f"Dataset: {n_samples} images, each {H}x{W} pixels")

---
## Section 2: Visualise Sample Images

Before any math, just see what we're working with.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle("Sample Images from Dataset", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(faces[i], cmap='gray')
    ax.set_title(f"img {i}", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## Section 3: SVD-Based Image Compression

### Theory

Any matrix $A \in \mathbb{R}^{m \times n}$ can be decomposed as:
$$A = U \Sigma V^T = \sum_{i=1}^{r} \sigma_i \, u_i v_i^T$$

where $r = \text{rank}(A)$, $\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_r > 0$.

The **rank-k approximation** keeps only the top-k terms:
$$A_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^T$$

By the **Eckart–Young theorem**, $A_k$ is the *best* rank-k approximation in Frobenius norm:
$$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \dots + \sigma_r^2}$$

**Compression ratio**: Original image needs $m \times n$ numbers. Rank-k approximation needs $k(m + n + 1)$ numbers.
$$\text{ratio} = \frac{k(m + n + 1)}{mn}$$

### Important Code: Computing SVD and Reconstruction

In [ ]:
# ─── IMPORTANT ─────────────────────────────────────────────────────────────
# np.linalg.svd decomposes A into U, s, Vt such that A = U @ diag(s) @ Vt
# full_matrices=False gives the "economy" SVD — shapes are:
#   U:  (m, r),  s: (r,),  Vt: (r, n)   where r = min(m,n)
# This avoids computing the huge zero-padded parts we don't need.
# ───────────────────────────────────────────────────────────────────────────

def svd_compress(image, k):
    """
    Compress a 2D grayscale image using rank-k SVD approximation.
    Returns the reconstructed image and compression ratio.
    """
    U, s, Vt = np.linalg.svd(image, full_matrices=False)
    
    # Rank-k reconstruction: keep only top-k singular triplets
    # U[:, :k]  — first k left singular vectors  (basis for column space)
    # s[:k]     — top k singular values           (importance weights)
    # Vt[:k, :] — first k right singular vectors (basis for row space)
    A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    
    # Clip to valid pixel range [0, 1]
    A_k = np.clip(A_k, 0, 1)
    
    m, n = image.shape
    compression_ratio = k * (m + n + 1) / (m * n)
    
    return A_k, compression_ratio, s


def reconstruction_error(original, reconstructed):
    """Frobenius norm error between original and reconstructed image."""
    return np.linalg.norm(original - reconstructed, 'fro')


# Choose a test image
test_image = faces[0]

# Compute SVD once and show singular value spectrum
U, s, Vt = np.linalg.svd(test_image, full_matrices=False)

print(f"Image shape: {test_image.shape}")
print(f"Number of singular values: {len(s)}")
print(f"Top 10 singular values: {np.round(s[:10], 3)}")
print(f"\nEnergy captured by top-k singular values:")
total_energy = np.sum(s**2)
for k in [5, 10, 20, 30, 50]:
    energy = np.sum(s[:k]**2) / total_energy * 100
    print(f"  k={k:3d}: {energy:.1f}% of total energy")

### Visualise Compression at Different k Values

In [ ]:
k_values = [1, 5, 10, 20, 40, 64]

fig, axes = plt.subplots(2, len(k_values) + 1, figsize=(18, 6))
fig.suptitle("SVD Image Compression — Effect of Rank k", fontsize=14, fontweight='bold')

# Row 0: images
axes[0, 0].imshow(test_image, cmap='gray')
axes[0, 0].set_title("Original", fontsize=10, fontweight='bold')
axes[0, 0].axis('off')

# Row 1: difference maps
axes[1, 0].imshow(np.zeros_like(test_image), cmap='hot', vmin=0, vmax=0.3)
axes[1, 0].set_title("Error Map", fontsize=9)
axes[1, 0].axis('off')

for i, k in enumerate(k_values):
    reconstructed, ratio, _ = svd_compress(test_image, k)
    err = reconstruction_error(test_image, reconstructed)
    
    axes[0, i+1].imshow(reconstructed, cmap='gray')
    axes[0, i+1].set_title(f"k={k}\n{ratio*100:.1f}% size", fontsize=9)
    axes[0, i+1].axis('off')
    
    # Error map — shows where information was lost
    diff = np.abs(test_image - reconstructed)
    axes[1, i+1].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
    axes[1, i+1].set_title(f"err={err:.3f}", fontsize=9)
    axes[1, i+1].axis('off')

plt.tight_layout()
plt.savefig('/home/claude/svd_compression.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: svd_compression.png")

### Singular Value Spectrum and Error vs k

In [ ]:
# ─── IMPORTANT ─────────────────────────────────────────────────────────────
# The singular value spectrum tells us how "compressible" an image is.
# A fast drop → image has low intrinsic rank → high compressibility.
# A slow drop → image has complex structure → needs more components.
# ───────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Singular value spectrum
axes[0].semilogy(s, 'b-o', markersize=3)
axes[0].set_title("Singular Value Spectrum", fontweight='bold')
axes[0].set_xlabel("Index i")
axes[0].set_ylabel("σᵢ (log scale)")
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative energy captured
cumulative_energy = np.cumsum(s**2) / np.sum(s**2) * 100
axes[1].plot(cumulative_energy, 'g-')
axes[1].axhline(y=90, color='r', linestyle='--', label='90% energy')
axes[1].axhline(y=99, color='orange', linestyle='--', label='99% energy')
k_90 = np.argmax(cumulative_energy >= 90) + 1
k_99 = np.argmax(cumulative_energy >= 99) + 1
axes[1].axvline(x=k_90, color='r', linestyle=':', alpha=0.7)
axes[1].axvline(x=k_99, color='orange', linestyle=':', alpha=0.7)
axes[1].set_title(f"Cumulative Energy\n(90%→k={k_90}, 99%→k={k_99})", fontweight='bold')
axes[1].set_xlabel("k (number of components)")
axes[1].set_ylabel("% Energy Captured")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Reconstruction error vs k
k_range = list(range(1, 65, 2))
errors = []
ratios = []
for k in k_range:
    rec, ratio, _ = svd_compress(test_image, k)
    errors.append(reconstruction_error(test_image, rec))
    ratios.append(ratio * 100)

axes[2].plot(k_range, errors, 'r-o', markersize=3, label='Frob. Error')
ax2 = axes[2].twinx()
ax2.plot(k_range, ratios, 'b--s', markersize=3, alpha=0.6, label='Storage %')
axes[2].set_title("Error & Storage vs k", fontweight='bold')
axes[2].set_xlabel("k")
axes[2].set_ylabel("Reconstruction Error", color='r')
ax2.set_ylabel("Storage Used (%)", color='b')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/svd_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: svd_analysis.png")

---
## Section 4: PCA and Eigenfaces — Pattern Detection

### Theory

Given $N$ face images, each flattened to a vector $x_i \in \mathbb{R}^d$ (where $d = H \times W$):

**Step 1 — Mean-center the data:**
$$\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i, \qquad \tilde{x}_i = x_i - \bar{x}$$

**Step 2 — Form the data matrix and covariance matrix:**
$$X = [\tilde{x}_1 \mid \tilde{x}_2 \mid \dots \mid \tilde{x}_N]^T \in \mathbb{R}^{N \times d}$$
$$C = \frac{1}{N} X^T X \in \mathbb{R}^{d \times d}$$

**Step 3 — Eigendecomposition of C:**
$$C = Q \Lambda Q^T$$
Eigenvectors $q_i$ (columns of $Q$) = **eigenfaces**. They form an orthonormal basis.

**Step 4 — Orthogonal projection onto top-k eigenfaces:**
$$\hat{x} = \bar{x} + Q_k Q_k^T \tilde{x}$$
This is the best rank-k approximation of $x$ in the eigenface subspace.

**Key insight**: SVD of $X$ directly gives the eigenfaces as right singular vectors. This is more numerically stable than computing $C = X^TX$ explicitly.

### Important Code: Computing Eigenfaces via SVD of Data Matrix

In [ ]:
# ─── IMPORTANT ─────────────────────────────────────────────────────────────
# We flatten each image (64x64) → (4096,) to form the data matrix X.
# Then we:
#   1. Mean-center X (subtract the mean face)
#   2. Take SVD of X (NOT eigendecomposition of X^TX — numerically better)
#   3. Right singular vectors Vt.T = eigenfaces (principal directions)
#   4. Eigenvalues of C = σᵢ² / N (variance explained by each eigenface)
# ───────────────────────────────────────────────────────────────────────────

# Step 1: Flatten images → data matrix
X_flat = faces.reshape(n_samples, -1)   # shape: (400, 4096)
print(f"Data matrix shape: {X_flat.shape}")

# Step 2: Mean-center
mean_face = X_flat.mean(axis=0)          # shape: (4096,)
X_centered = X_flat - mean_face          # subtract mean face from every image

# Step 3: SVD of centered data matrix
# X = U S Vt
# Columns of Vt.T = right singular vectors = eigenfaces (principal components)
# We only need the first min(N,d) components → full_matrices=False
U_pca, s_pca, Vt_pca = np.linalg.svd(X_centered, full_matrices=False)

# Eigenfaces: each row of Vt_pca is one eigenface, reshaped to (64,64)
eigenfaces = Vt_pca.reshape(-1, H, W)   # shape: (400, 64, 64)

# Variance explained by each component
variance_explained = (s_pca ** 2) / np.sum(s_pca ** 2)

print(f"\nEigenfaces computed: {eigenfaces.shape}")
print(f"Variance explained by top components:")
for i in range(10):
    print(f"  PC{i+1}: {variance_explained[i]*100:.2f}%")
print(f"  ...")
print(f"  Top-20 total: {variance_explained[:20].sum()*100:.1f}%")
print(f"  Top-50 total: {variance_explained[:50].sum()*100:.1f}%")

### Visualise the Mean Face and Top Eigenfaces

In [ ]:
n_eigenfaces_to_show = 15

fig = plt.figure(figsize=(18, 5))
fig.suptitle("Mean Face and Top Eigenfaces", fontsize=14, fontweight='bold')

# Mean face
ax = fig.add_subplot(2, 9, 1)
ax.imshow(mean_face.reshape(H, W), cmap='gray')
ax.set_title("Mean Face", fontsize=9, fontweight='bold')
ax.axis('off')

# Top eigenfaces
for i in range(n_eigenfaces_to_show):
    ax = fig.add_subplot(2, 9, i + 2 if i < 8 else i + 3)
    ef = eigenfaces[i]
    # Normalize for display (eigenfaces can have negative values)
    ef_display = (ef - ef.min()) / (ef.max() - ef.min())
    ax.imshow(ef_display, cmap='gray')
    ax.set_title(f"EF {i+1}\n({variance_explained[i]*100:.1f}%)", fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/home/claude/eigenfaces.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eigenfaces.png")

---
## Section 5: Orthogonal Projection — Reconstruct Faces from Eigenfaces

### Theory

To reconstruct face $x$ using the top-k eigenfaces $\{q_1, \dots, q_k\}$:

1. **Project** onto eigenface subspace: $\alpha_i = q_i^T(x - \bar{x})$ — these are the **coordinates** (PCA scores)
2. **Reconstruct**: $\hat{x} = \bar{x} + \sum_{i=1}^{k} \alpha_i q_i = \bar{x} + Q_k Q_k^T (x - \bar{x})$

$Q_k Q_k^T$ is an **orthogonal projection matrix** onto the subspace spanned by the top-k eigenfaces. This is the LA core of the project.

In [ ]:
# ─── IMPORTANT ─────────────────────────────────────────────────────────────
# reconstruct_face() performs an orthogonal projection:
#   1. Take centered face: x_c = x - mean_face
#   2. Project onto k-dimensional eigenface subspace:
#      coords = Qk.T @ x_c   (coordinates in eigenface space, shape (k,))
#   3. Reconstruct: x_hat = mean_face + Qk @ coords
#   This is equivalent to x_hat = mean_face + (Qk Qk^T) x_c
#   where Qk Qk^T is the orthogonal projection matrix onto the subspace.
# ───────────────────────────────────────────────────────────────────────────

def reconstruct_face(face_flat, eigenfaces_flat, mean_face_flat, k):
    """
    Reconstruct a face using top-k eigenfaces (orthogonal projection).
    
    face_flat:       (d,) flattened face image
    eigenfaces_flat: (n_components, d) eigenfaces as row vectors
    mean_face_flat:  (d,) mean face
    k:               number of eigenfaces to use
    """
    Qk = eigenfaces_flat[:k].T           # shape: (d, k)  — basis vectors as columns
    x_centered = face_flat - mean_face_flat
    
    # Orthogonal projection: project x_centered onto column space of Qk
    coords = Qk.T @ x_centered            # shape: (k,)  — PCA scores
    x_reconstructed = Qk @ coords         # shape: (d,)  — back to image space
    
    return np.clip(mean_face_flat + x_reconstructed, 0, 1)


# Test on a few faces
eigenfaces_flat = Vt_pca                  # shape: (n_components, d)
k_values_pca = [1, 5, 10, 20, 50, 100]
test_indices = [0, 5, 15]                 # which faces to reconstruct

fig, axes = plt.subplots(
    len(test_indices), len(k_values_pca) + 1,
    figsize=(16, 5)
)
fig.suptitle("Face Reconstruction via Orthogonal Projection onto Eigenface Subspace",
             fontsize=13, fontweight='bold')

for row, idx in enumerate(test_indices):
    face_flat = X_flat[idx]
    
    # Original
    axes[row, 0].imshow(face_flat.reshape(H, W), cmap='gray')
    axes[row, 0].set_title("Original" if row == 0 else "", fontsize=9)
    axes[row, 0].set_ylabel(f"Face {idx}", fontsize=9)
    axes[row, 0].axis('off')
    
    for col, k in enumerate(k_values_pca):
        reconstructed = reconstruct_face(face_flat, eigenfaces_flat, mean_face, k)
        err = np.linalg.norm(face_flat - reconstructed)
        
        axes[row, col + 1].imshow(reconstructed.reshape(H, W), cmap='gray')
        if row == 0:
            axes[row, col + 1].set_title(f"k={k}", fontsize=9)
        axes[row, col + 1].set_xlabel(f"err={err:.2f}", fontsize=8)
        axes[row, col + 1].axis('off')

plt.tight_layout()
plt.savefig('/home/claude/face_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: face_reconstruction.png")

---
## Section 6: Variance Explained and Reconstruction Error vs k

In [ ]:
# Compute reconstruction errors across all test faces for different k
k_range_pca = list(range(1, 101, 2))
mean_errors = []

for k in k_range_pca:
    errs = [
        np.linalg.norm(X_flat[i] - reconstruct_face(X_flat[i], eigenfaces_flat, mean_face, k))
        for i in range(min(50, n_samples))   # use first 50 for speed
    ]
    mean_errors.append(np.mean(errs))

cumvar = np.cumsum(variance_explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("PCA Analysis: Variance Explained vs Reconstruction Error",
             fontsize=13, fontweight='bold')

# Plot 1: Variance explained
axes[0].bar(range(1, 31), variance_explained[:30] * 100, color='steelblue', alpha=0.7)
ax_cum = axes[0].twinx()
ax_cum.plot(range(1, 31), cumvar[:30] * 100, 'r-o', markersize=4, label='Cumulative')
ax_cum.axhline(90, color='orange', linestyle='--', alpha=0.7, label='90%')
axes[0].set_title("Variance Explained per Eigenface", fontweight='bold')
axes[0].set_xlabel("Eigenface index")
axes[0].set_ylabel("Variance Explained (%)", color='steelblue')
ax_cum.set_ylabel("Cumulative (%)", color='r')
ax_cum.legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Reconstruction error vs k
axes[1].plot(k_range_pca, mean_errors, 'purple', linewidth=2)
axes[1].fill_between(k_range_pca, mean_errors, alpha=0.2, color='purple')
axes[1].set_title("Mean Reconstruction Error vs k", fontweight='bold')
axes[1].set_xlabel("k (number of eigenfaces)")
axes[1].set_ylabel("Mean L2 Error")
axes[1].grid(True, alpha=0.3)

# Annotate elbow
k_90 = np.argmax(cumvar >= 0.90) + 1
axes[1].axvline(k_90, color='r', linestyle='--', label=f'k={k_90} (90% var)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/claude/pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pca_analysis.png")

---
## Section 7: Connecting SVD and Eigendecomposition

### The Mathematical Link (Important for Report)

| Concept | SVD form | Eigendecomposition form |
|---|---|---|
| Image compression | $A = U\Sigma V^T$, keep top-k | — |
| PCA (eigenfaces) | SVD of $X$: right singular vectors = eigenfaces | Eigenvectors of $C = X^TX/N$ |
| Singular values | $\sigma_i$ of $X$ | $\lambda_i = \sigma_i^2/N$ (eigenvalues of $C$) |
| Projection | $A_k = U_k\Sigma_k V_k^T$ | $\hat{x} = Q_k Q_k^T x$ |

**They are the same computation** — SVD of $X$ is the preferred numerical path to PCA.

In [ ]:
# ─── IMPORTANT ─────────────────────────────────────────────────────────────
# VERIFY: eigenvalues of covariance matrix == (singular values of X)^2 / N
# This confirms SVD and eigendecomposition give identical results.
# ───────────────────────────────────────────────────────────────────────────

# Method 1: SVD of data matrix (what we've been doing)
eigenvalues_from_svd = s_pca[:10]**2 / n_samples

# Method 2: Direct eigendecomposition of small covariance (N x N trick)
# For N < d, compute eigenvectors via N×N matrix (much faster!)
# C_small = X_centered @ X_centered.T / N  (N×N, not d×d)
C_small = (X_centered @ X_centered.T) / n_samples
eigenvalues_direct = np.sort(np.linalg.eigvalsh(C_small))[::-1]  # descending

print("Verification: SVD vs Eigendecomposition")
print(f"{'Eigenvalue':>12} {'via SVD':>12} {'via EigenDecomp':>16} {'Match':>8}")
print("-" * 52)
for i in range(10):
    match = abs(eigenvalues_from_svd[i] - eigenvalues_direct[i]) < 1e-6
    print(f"{i+1:>12} {eigenvalues_from_svd[i]:>12.4f} {eigenvalues_direct[i]:>16.4f} {'✓' if match else '✗':>8}")

print("\n→ Both methods give identical eigenvalues. SVD is preferred numerically.")

---
## Section 8: Final Summary

| | Image Compression (SVD) | Pattern Detection (Eigenfaces) |
|---|---|---|
| **Input** | Single image matrix $A$ | Dataset of $N$ face images |
| **Decomposition** | SVD of $A$ | SVD of centered data matrix $X$ |
| **Key vectors** | Left/right singular vectors | Right singular vectors = eigenfaces |
| **Key values** | Singular values $\sigma_i$ | Eigenvalues $\lambda_i = \sigma_i^2/N$ |
| **Projection** | Rank-k approx of image | Projection onto k-eigenface subspace |
| **Error bound** | $\|A-A_k\|_F = \sqrt{\sum_{i>k}\sigma_i^2}$ | $\|x - \hat{x}\|^2 = \sum_{i>k}\lambda_i$ |
| **Real-world use** | JPEG compression, streaming | Face recognition, biometrics |

### Key Takeaways

1. **SVD is the engine** behind both compression and PCA/eigenfaces
2. **Orthogonal projections** (onto singular/eigenvector subspaces) give the *best* approximation in Frobenius norm (Eckart–Young)
3. **Singular values decay fast** → real-world images are low-rank in disguise → high compressibility
4. **Eigenfaces are a basis** for the "face space" — any face is approximately a linear combination of eigenfaces

In [ ]:
print("Project Complete!")
print("\nFiles generated:")
print("  svd_compression.png   — compression at different k values")
print("  svd_analysis.png      — singular value spectrum + error analysis")
print("  eigenfaces.png        — mean face + top eigenfaces")
print("  face_reconstruction.png — orthogonal projection reconstructions")
print("  pca_analysis.png      — variance explained + reconstruction error")